In [1]:
# =====================================
# Transformer V2 - Imports & Dataset
# =====================================

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding,
    Add
)

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("TensorFlow Version:", tf.__version__)

# Dataset path
DATA_PATH = os.path.join("..", "dataset", "training")

# Load datasets
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("\n✅ Dataset Loaded Successfully!")
print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

TensorFlow Version: 2.21.0

✅ Dataset Loaded Successfully!
Train      : (7608, 154, 63) (7608,)
Validation : (275, 154, 63) (275,)
Test       : (487, 154, 63) (487,)


In [2]:
# =====================================
# Dataset Check + Normalization
# =====================================

print("========== DATASET CHECK ==========")

print("Train shape :", X_train.shape, y_train.shape)
print("Val shape   :", X_val.shape, y_val.shape)
print("Test shape  :", X_test.shape, y_test.shape)

print("\nTrain labels:", len(np.unique(y_train)))
print("Val labels  :", len(np.unique(y_val)))
print("Test labels :", len(np.unique(y_test)))

print("\nLabel range (train):", y_train.min(), "to", y_train.max())

# -------- Normalize --------

train_mean = np.mean(X_train)
train_std = np.std(X_train) + 1e-8

X_train = (X_train - train_mean) / train_std
X_val   = (X_val - train_mean) / train_std
X_test  = (X_test - train_mean) / train_std

print("\n✅ Normalization Complete!")
print("Train mean :", np.mean(X_train))
print("Train std  :", np.std(X_train))
print("Val mean   :", np.mean(X_val))
print("Test mean  :", np.mean(X_test))

num_classes = len(np.unique(y_train))
print("\nNumber of classes:", num_classes)

========== DATASET CHECK ==========
Train shape : (7608, 154, 63) (7608,)
Val shape   : (275, 154, 63) (275,)
Test shape  : (487, 154, 63) (487,)

Train labels: 210
Val labels  : 210
Test labels : 210

Label range (train): 0 to 209

✅ Normalization Complete!
Train mean : -5.217071e-07
Train std  : 0.99999875
Val mean   : -0.0042453213
Test mean  : 0.0033953818

Number of classes: 210


In [3]:
# =====================================
# Build Transformer V2
# =====================================

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding,
    Add
)
from tensorflow.keras.models import Model
import tensorflow as tf

SEQUENCE_LENGTH = 154
FEATURE_DIM = 63
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256
DROPOUT_RATE = 0.3


def transformer_encoder(inputs):
    # Multi-head Self Attention
    attention = MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=EMBED_DIM // NUM_HEADS,
        dropout=DROPOUT_RATE
    )(inputs, inputs)

    x = Add()([inputs, attention])
    x = LayerNormalization(epsilon=1e-6)(x)

    # Feed Forward Network
    ff = Dense(FF_DIM, activation="relu")(x)
    ff = Dropout(DROPOUT_RATE)(ff)
    ff = Dense(EMBED_DIM)(ff)

    x = Add()([x, ff])
    x = LayerNormalization(epsilon=1e-6)(x)

    return x


# ---------- Input ----------
inputs = Input(shape=(SEQUENCE_LENGTH, FEATURE_DIM))

# Project MediaPipe features to embedding space
x = Dense(EMBED_DIM)(inputs)

# Positional Embeddings (VERY IMPORTANT)
positions = tf.range(start=0, limit=SEQUENCE_LENGTH, delta=1)
position_embeddings = Embedding(
    input_dim=SEQUENCE_LENGTH,
    output_dim=EMBED_DIM
)(positions)

x = x + position_embeddings

# Two Transformer Encoder Blocks
x = transformer_encoder(x)
x = transformer_encoder(x)

# Classification Head
x = GlobalAveragePooling1D()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

outputs = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=inputs, outputs=outputs)

print("✅ Transformer V2 built successfully!")

✅ Transformer V2 built successfully!


In [4]:
# ==========================================
# Build Transformer Model
# ==========================================

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model

# Input shape
input_shape = (154, 63)

inputs = Input(shape=input_shape)

# Project 63 features to 128 dimensions
x = Dense(128)(inputs)

# Learnable positional embedding
positions = tf.range(start=0, limit=input_shape[0], delta=1)
position_embedding = tf.keras.layers.Embedding(
    input_dim=input_shape[0],
    output_dim=128
)(positions)

x = x + position_embedding

# -------- Transformer Block 1 --------
attn = MultiHeadAttention(
    num_heads=4,
    key_dim=32,
    dropout=0.1
)(x, x)

x = LayerNormalization(epsilon=1e-6)(x + attn)

ffn = Dense(256, activation="relu")(x)
ffn = Dropout(0.2)(ffn)
ffn = Dense(128)(ffn)

x = LayerNormalization(epsilon=1e-6)(x + ffn)

# -------- Transformer Block 2 --------
attn = MultiHeadAttention(
    num_heads=4,
    key_dim=32,
    dropout=0.1
)(x, x)

x = LayerNormalization(epsilon=1e-6)(x + attn)

ffn = Dense(256, activation="relu")(x)
ffn = Dropout(0.2)(ffn)
ffn = Dense(128)(ffn)

x = LayerNormalization(epsilon=1e-6)(x + ffn)

# Classification head
x = GlobalAveragePooling1D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

outputs = Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 154, 63)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 154, 128)  │      8,192 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 154, 128)  │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ add_5[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 154, 128)  │          0 │ add_5[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 154, 256)  │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 154, 256)  │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 154, 128)  │     32,896 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 316,754 (1.21 MB)

 Trainable params: 316,754 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# =====================================
# Compile Transformer V2
# =====================================

from tensorflow.keras.optimizers import Adam

optimizer = Adam(
    learning_rate=0.0002,   # Lower than BiLSTM (important!)
    clipnorm=1.0             # Stabilizes training
)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("✅ Transformer V2 compiled successfully!\n")

model.summary()

✅ Transformer V2 compiled successfully!



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 154, 63)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 154, 128)  │      8,192 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 154, 128)  │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ add_5[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 154, 128)  │          0 │ add_5[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 154, 256)  │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 154, 256)  │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 154, 128)  │     32,896 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 316,754 (1.21 MB)

 Trainable params: 316,754 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# =====================================
# Train Transformer V2
# =====================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

import os

os.makedirs("../models", exist_ok=True)

# Stop if validation loss stops improving
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when stuck
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# Save BEST transformer separately
checkpoint = ModelCheckpoint(
    "../models/best_transformer_v2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

Epoch 1/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step - accuracy: 0.0074 - loss: 5.3492
Epoch 1: val_accuracy improved from None to 0.01455, saving model to ../models/best_transformer_v2.keras

Epoch 1: finished saving model to ../models/best_transformer_v2.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 74s 286ms/step - accuracy: 0.0074 - loss: 5.3492 - val_accuracy: 0.0145 - val_loss: 5.2994 - learning_rate: 2.0000e-04
Epoch 2/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step - accuracy: 0.0124 - loss: 5.2321
Epoch 2: val_accuracy did not improve from 0.01455
238/238 ━━━━━━━━━━━━━━━━━━━━ 97s 409ms/step - accuracy: 0.0124 - loss: 5.2321 - val_accuracy: 0.0036 - val_loss: 5.1309 - learning_rate: 2.0000e-04
Epoch 3/50
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step - accuracy: 0.0170 - loss: 5.0785
Epoch 3: val_accuracy improved from 0.01455 to 0.04000, saving model to ../models/best_transformer_v2.keras

Epoch 3: finished saving model to ../models/best_transformer_v2.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 83s 3

In [7]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

print("="*35)
print(f"Transformer Test Accuracy : {test_acc*100:.2f}%")
print(f"Transformer Test Loss     : {test_loss:.4f}")
print("="*35)

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 115ms/step - accuracy: 0.7639 - loss: 0.9639
Transformer Test Accuracy : 76.39%
Transformer Test Loss     : 0.9639
